# Angle calculation

## Parameters

In [1]:
# --------------------
# Parameters / column names
# --------------------

INPUT_CENTROMERE_CSV = "C:/Users/u0119446/Desktop/Temp/Ongoing_projects/Ridha/dataset_ph3/processed/analysed/centromere.csv"
INPUT_LUMEN_CSV      = "C:/Users/u0119446/Desktop/Temp/Ongoing_projects/Ridha/dataset_ph3/processed/analysed/lumen_spots.csv"
OUTPUT_RESULTS_DIR   = "C:/Users/u0119446/Desktop/Temp/Ongoing_projects/Ridha/dataset_ph3/processed/analysed/"

CENT_FILENAME_COL = "Filename"
CENT_CELL_ID_COL  = "nuclei_id"
CENT_POINT_ID_COL = "centromere_id"
CENT_X_COL = "CentroidPxX"
CENT_Y_COL = "CentroidPxY"
CENT_Z_COL = "CentroidPxZ"

LUM_FILENAME_COL = "Filename"
LUM_POINT_ID_COL = "lumenspot_id"
LUM_X_COL = "CentroidPxX"
LUM_Y_COL = "CentroidPxY"
LUM_Z_COL = "CentroidPxZ"

# --- voxel scaling (update with your real units!) ---
VOXEL_SIZE_X = 0.17
VOXEL_SIZE_Y = 0.17
VOXEL_SIZE_Z = 0.25

# --- for the analysis ---
K_NEIGHBORS = 50
STRICT_TWO_CENTROMERES = True
GFP_THRESHOLD = 40 # This number was obtained from different negative nuclei presetent wihin image i6_prl01-10ul_a56_gfp_numa_ph3_40x_xy07

In [7]:
# --------------------
# Script 1: Generate kNN-plane dataframes (+ Filename metadata + mean_gfp/mean_pvim)
# --------------------

import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import cKDTree as KDTree

# --------------------
# Load data
# --------------------
cent = pd.read_csv(INPUT_CENTROMERE_CSV)
lum  = pd.read_csv(INPUT_LUMEN_CSV)

# --------------------
# If needed take out unnecesasary text added by NIS Elements to the filename column
# Substring to remove
substring = '01_lumen_segmentation_ridhasproject_withph3_000_'

# To remove all occurrences of the substring in both dataframes:
cent['Filename'] = cent['Filename'].str.replace(substring, '', regex=False)
lum['Filename'] = lum['Filename'].str.replace(substring, '', regex=False)

def to_phys(df, xcol, ycol, zcol):
    """Convert pixel coords to physical units using voxel sizes."""
    return np.column_stack([
        df[xcol].to_numpy() * VOXEL_SIZE_X,
        df[ycol].to_numpy() * VOXEL_SIZE_Y,
        df[zcol].to_numpy() * VOXEL_SIZE_Z,
    ])

cent["_phys_xyz"] = list(to_phys(cent, CENT_X_COL, CENT_Y_COL, CENT_Z_COL))
lum["_phys_xyz"]  = list(to_phys(lum,  LUM_X_COL,  LUM_Y_COL,  LUM_Z_COL))

# --------------------
# Helpers
# --------------------
def fit_plane_pca(points):
    """Fit a plane to points using PCA (SVD)."""
    P = np.asarray(points, dtype=float)
    C = P.mean(axis=0)
    Q = P - C
    U, S, Vt = np.linalg.svd(Q, full_matrices=False)
    n = Vt[-1, :]
    n = n / np.linalg.norm(n)
    dists = Q @ n
    rms = float(np.sqrt(np.mean(dists**2)))
    eigvals = (S**2) / max(len(P) - 1, 1)
    return C, n, rms, np.sort(eigvals)

def acute_angle_deg(v1, v2, eps=1e-12):
    """Angle (0–90°) between two vectors."""
    a, b = np.asarray(v1,float), np.asarray(v2,float)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < eps or nb < eps:
        return np.nan
    cosang = np.clip(np.abs(np.dot(a, b) / (na * nb)), -1.0, 1.0)
    return float(np.degrees(np.arccos(cosang)))

def _filename_parts(fname):
    """
    Return underscore-split parts of the file *stem* (without extension),
    e.g., 'i6_pr1301_a35_gfp_numa_pvim_40X_xy11.nd2' -> 
    ['i6','pr1301','a35','gfp','numa','pvim','40X','xy11']
    """
    base = os.path.basename(str(fname))
    stem = os.path.splitext(base)[0]
    return stem.split("_")

def _extract_meta(fname):
    """
    Safely extract repeat, condition, dox, roi from underscore-split Filename.
    Missing parts -> None.
    """
    parts = _filename_parts(fname)
    get = lambda i: parts[i] if i < len(parts) else None
    return {
        "repeat":    get(0),  # element 0
        "condition": get(1),  # element 1
        "dox":       get(2),  # element 2
        "roi":       get(7),  # element 7 (e.g., xy11)
    }

# --------------------
# Build KD-trees per filename
# --------------------
trees = {}
lum_arrays = {}
for fname, sub in lum.groupby(LUM_FILENAME_COL):
    arr = np.vstack(sub["_phys_xyz"].to_numpy())
    lum_arrays[fname] = (arr, sub.reset_index(drop=True))
    trees[fname] = KDTree(arr)

# --------------------
# Process per nucleus
# --------------------
rows_long = []
rows_summary = []
skipped = []

for (fname, cell_id), g in cent.groupby([CENT_FILENAME_COL, CENT_CELL_ID_COL]):
    if STRICT_TWO_CENTROMERES and len(g) != 2:
        skipped.append((fname, int(cell_id), len(g)))
        continue

    g = g.sort_values(CENT_POINT_ID_COL)
    c_xyz = np.vstack(g["_phys_xyz"].to_numpy())
    c_ids = g[CENT_POINT_ID_COL].tolist()
    v = c_xyz[1] - c_xyz[0]

    if fname not in trees:
        skipped.append((fname, int(cell_id), "no_lumen"))
        continue

    tree = trees[fname]
    lum_arr, lum_df = lum_arrays[fname]

    dists, idxs = tree.query(c_xyz, k=1)
    chosen_idx = 0 if dists[0] <= dists[1] else 1
    chosen_cent_id = int(c_ids[chosen_idx])
    nearest_dist = float(dists[chosen_idx])
    nearest_idx = int(idxs[chosen_idx])
    p0 = lum_arr[nearest_idx]

    k = min(K_NEIGHBORS, len(lum_arr))
    knn_dists, knn_idxs = tree.query(p0, k=k)
    knn_points = lum_arr[np.atleast_1d(knn_idxs).astype(int)]

    centroid, normal, rms, eigvals = fit_plane_pca(knn_points)
    flatness_fraction = float(eigvals[0] / max(eigvals[-1], 1e-12))
    angle_deg = 90 - float(acute_angle_deg(normal, v))  # 0° when division axis is parallel to plane

    # ---- NEW: per-cell means for GFP & PVIM (averaged over the two centromere rows)
    # Change the column names below if your CSV uses different ones.
    mean_gfp  = float(g["mean_gfp"].mean())  if "mean_gfp"  in g.columns else np.nan
    mean_pvim = float(g["mean_ph3"].mean()) if "mean_pvim" in g.columns else np.nan

    # Closest distance to lumen per nucleus again averaged between the two centromeres but it is just the same value
    nearest_dist = float(g["nuclei_nndtolumen_bordertoborder"].mean()) if "nuclei_nndtolumen_bordertoborder" in g.columns else np.nan

    gfp_positive = mean_gfp > GFP_THRESHOLD
    close_to_lumen = nearest_dist < 10.0  # microns

    # ---- NEW: Filename metadata (repeat, condition, dox, roi)
    meta = _extract_meta(fname)

    rows_summary.append({
        "Filename": fname,
        "repeat":    meta["repeat"],
        "condition": meta["condition"],
        "dox":       meta["dox"],
        "roi":       meta["roi"],
        "nuclei_id": int(cell_id),
        "centromere1_id": int(c_ids[0]),
        "centromere2_id": int(c_ids[1]),
        "chosen_centromere_id": chosen_cent_id,
        "chosen_lumenspot_id" : nearest_idx,
        "centromere1_x": c_xyz[0,0], "centromere1_y": c_xyz[0,1], "centromere1_z": c_xyz[0,2],
        "centromere2_x": c_xyz[1,0], "centromere2_y": c_xyz[1,1], "centromere2_z": c_xyz[1,2],
        "division_axis_len": np.linalg.norm(v),
        "nearest_lumen_distance": nearest_dist,
        "k_neighbors": k,
        "plane_cx": centroid[0], "plane_cy": centroid[1], "plane_cz": centroid[2],
        "plane_nx": normal[0],   "plane_ny": normal[1],   "plane_nz": normal[2],
        "angle_deg": angle_deg,
        "plane_fit_rms": rms,
        "flatness_fraction": flatness_fraction,
        "mean_gfp":  mean_gfp,
        "mean_ph3": mean_pvim,
        "gfp_positive": gfp_positive,
        "close_to_lumen": close_to_lumen,
    })

    for rank, li in enumerate(np.atleast_1d(knn_idxs)):
        px,py,pz = lum_arr[int(li)]
        rows_long.append({
            "Filename": fname,
            "nuclei_id": int(cell_id),
            "centromere_pair": f"{c_ids[0]}-{c_ids[1]}",
            "chosen_centromere_id": chosen_cent_id,
            "neighbor_rank": int(rank),
            "neighbor_x": px, "neighbor_y": py, "neighbor_z": pz,
            "centromere1_x": c_xyz[0,0], "centromere1_y": c_xyz[0,1], "centromere1_z": c_xyz[0,2],
            "centromere2_x": c_xyz[1,0], "centromere2_y": c_xyz[1,1], "centromere2_z": c_xyz[1,2],
            "chosen_centromere_x": c_xyz[chosen_idx,0],
            "chosen_centromere_y": c_xyz[chosen_idx,1],
            "chosen_centromere_z": c_xyz[chosen_idx,2],
            "plane_cx": centroid[0], "plane_cy": centroid[1], "plane_cz": centroid[2],
            "plane_nx": normal[0],   "plane_ny": normal[1],   "plane_nz": normal[2],
            "angle_deg": angle_deg, 
            "nearest_lumen_distance": nearest_dist,
            "plane_fit_rms": rms,
            "flatness_fraction": flatness_fraction,
        })

df_long = pd.DataFrame(rows_long).sort_values(["Filename","nuclei_id","neighbor_rank"]).reset_index(drop=True)
df_summary = pd.DataFrame(rows_summary).sort_values(["Filename","nuclei_id"]).reset_index(drop=True)

# Save outputs
Path(OUTPUT_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
df_long.to_csv(os.path.join(OUTPUT_RESULTS_DIR, "knn_plane_points.csv"), index=False)
df_summary.to_csv(os.path.join(OUTPUT_RESULTS_DIR, "knn_plane_summary.csv"), index=False)

print("Saved:")
print(" - knn_plane_points.csv (long-form, one row per neighbor)")
print(" - knn_plane_summary.csv (one row per nucleus, +metadata +mean_gfp/mean_pvim)")
print(f"Skipped {len(skipped)} cells (not exactly 2 centromeres or missing lumens).")


Saved:
 - knn_plane_points.csv (long-form, one row per neighbor)
 - knn_plane_summary.csv (one row per nucleus, +metadata +mean_gfp/mean_pvim)
Skipped 4 cells (not exactly 2 centromeres or missing lumens).


# Visualization

In [2]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import os
import numpy as np
import pandas as pd

df_long = pd.read_csv("D:/Ongoing_projects/user_projects/Ridha/knn_plane_points.csv")
df_summary = pd.read_csv("D:/Ongoing_projects/user_projects/Ridha/knn_plane_summary.csv")

def plot_cell_plotly(
    fname, nuclei_id, df_long, df_summary,
    plane_size=10,
    normalofplane_size=10,
    recenter_chosen_on_plane=False,
    anchor="chosen"  # "chosen" or "midpoint"
):
    """
    Plot one nucleus in WORLD coordinates with Plotly.
    If recenter_chosen_on_plane=True:
      - Reposition the centromere pair ONLY for visualization:
        * anchor="chosen": put the chosen centromere at the plane centroid,
                          keep the original division vector (angle preserved).
        * anchor="midpoint": put the midpoint of the pair at the plane centroid,
                             keep the original division vector (angle preserved).

    Everything else (lumen neighbors, plane, normal) stays in original coordinates.
    """

    # --- Slice data
    dfL = df_long[(df_long["Filename"]==fname) & (df_long["nuclei_id"]==nuclei_id)]
    if dfL.empty:
        print("No neighbors found for this nucleus.")
        return
    rowS = df_summary[(df_summary["Filename"]==fname) & (df_summary["nuclei_id"]==nuclei_id)].iloc[0]

    # Centromeres (world coords)
    c1 = np.array([rowS.centromere1_x, rowS.centromere1_y, rowS.centromere1_z], float)
    c2 = np.array([rowS.centromere2_x, rowS.centromere2_y, rowS.centromere2_z], float)
    v  = c2 - c1  # division vector (direction+length)
    chosen_id = int(rowS.chosen_centromere_id)
    chosen_is_c1 = (chosen_id == int(rowS.centromere1_id))
    chosen = c1 if chosen_is_c1 else c2

    # Plane centroid & normal (world)
    pc = np.array([rowS.plane_cx, rowS.plane_cy, rowS.plane_cz], float)
    n  = np.array([rowS.plane_nx, rowS.plane_ny, rowS.plane_nz], float)

    # Lumen neighbors (world)
    neigh = dfL[["neighbor_x","neighbor_y","neighbor_z"]].to_numpy(float)

    # --- Optionally re-center centromeres (translation only; angle preserved)
    if recenter_chosen_on_plane:
        if anchor == "chosen":
            # Put chosen at plane center; keep vector v exactly
            if chosen_is_c1:
                c1p = pc.copy()
                c2p = pc + v
            else:
                c2p = pc.copy()
                c1p = pc - v
        elif anchor == "midpoint":
            # Put midpoint at plane center; keep vector v
            mid = (c1 + c2) * 0.5
            t = pc - mid
            c1p = c1 + t
            c2p = c2 + t
        else:
            raise ValueError("anchor must be 'chosen' or 'midpoint'")
    else:
        # No recentering: use original world positions
        c1p, c2p = c1, c2

    # Recompute chosen point in case it moved
    chosenp = c1p if chosen_is_c1 else c2p

    # --- Build plane patch (world coords)
    vtemp = np.array([1,0,0], float)
    if np.allclose(np.cross(vtemp, n), 0.0):
        vtemp = np.array([0,1,0], float)
    v1 = np.cross(n, vtemp); v1 /= np.linalg.norm(v1)
    v2 = np.cross(n, v1)
    grid = np.linspace(-plane_size, plane_size, 5)
    Xg, Yg = np.meshgrid(grid, grid)
    Xp = pc[0] + v1[0]*Xg + v2[0]*Yg
    Yp = pc[1] + v1[1]*Xg + v2[1]*Yg
    Zp = pc[2] + v1[2]*Xg + v2[2]*Yg

    # --- Plotly figure
    fig = go.Figure()

    # Lumen neighbors
    fig.add_trace(go.Scatter3d(
        x=neigh[:,0], y=neigh[:,1], z=neigh[:,2],
        mode="markers",
        marker=dict(size=3, color="blue", opacity=0.6),
        name="kNN lumen"
    ))

    # Centromeres (possibly re-centered)
    fig.add_trace(go.Scatter3d(
        x=[c1p[0]], y=[c1p[1]], z=[c1p[2]],
        mode="markers", marker=dict(size=6, color="red"),
        name=f"Centromere {int(rowS.centromere1_id)}"
    ))
    fig.add_trace(go.Scatter3d(
        x=[c2p[0]], y=[c2p[1]], z=[c2p[2]],
        mode="markers", marker=dict(size=6, color="green"),
        name=f"Centromere {int(rowS.centromere2_id)}"
    ))
    fig.add_trace(go.Scatter3d(
        x=[chosenp[0]], y=[chosenp[1]], z=[chosenp[2]],
        mode="markers", marker=dict(size=10, color="yellow", line=dict(color="black", width=2)),
        name="Chosen centromere"
    ))

    # Division axis (uses possibly re-centered c1p, c2p)
    fig.add_trace(go.Scatter3d(
        x=[c1p[0], c2p[0]], y=[c1p[1], c2p[1]], z=[c1p[2], c2p[2]],
        mode="lines", line=dict(color="black", width=4), name="Division axis"
    ))

    # Plane patch (world coords)
    fig.add_trace(go.Surface(
        x=Xp, y=Yp, z=Zp,
        colorscale=[[0, "cyan"], [1, "cyan"]],
        opacity=0.3, showscale=False, name="Plane"
    ))

    # Plane normal (arrow line, world coords)
    fig.add_trace(go.Scatter3d(
        x=[pc[0] - n[0]*normalofplane_size, pc[0] + n[0]*normalofplane_size],
        y=[pc[1] - n[1]*normalofplane_size, pc[1] + n[1]*normalofplane_size],
        z=[pc[2] - n[2]*normalofplane_size, pc[2] + n[2]*normalofplane_size],
        mode="lines",
        line=dict(color="purple", width=6),
        name="Plane normal (±)"
    ))

    fig.update_layout(
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
        title=f"{fname} | Cell {nuclei_id} | Angle={rowS.angle_deg:.1f}°"
    )
    fig.show()

  
# chosen anchored at plane center (visual aid)
# plot_cell_plotly("20250813_nisv6.20.0_ridha_full_segmentation_v2_000_i6_pr1301_a35_gfp_numa_pvim_40X_xy14.nd2", 9, df_long, df_summary,
#                 recenter_chosen_on_plane=True, anchor="chosen")

plot_cell_plotly("i6_prl01-10ul_a35_gfp_numa_pvim_40X_xy9.nd2", 158, df_long, df_summary,
                 recenter_chosen_on_plane=False, anchor="chosen")

## Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

angles_deg = np.random.uniform(0, 90, 100)  # Replace with your data
angles_rad = np.deg2rad(angles_deg)

# Histogram: bin edges in radians
num_bins = 9
bin_edges = np.linspace(0, np.pi/2, num_bins + 1)
counts, _ = np.histogram(angles_rad, bins=bin_edges)

# Compute bin centers for polar bars
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
bars = ax.bar(bin_centers, counts, width=(np.pi/2)/num_bins, bottom=0, align='center')

ax.set_thetamin(0)
ax.set_thetamax(90)
#ax.set_theta_direction(1)  # If you want the orientation to match your example

plt.show()